# 🔥 নিজের data দিয়ে Data Science শেখা — BASS-II Flame Spread
**AgniShikha · Prachya Biswas**

এই notebook-এ তুমি নিজের হাতে সেই কাজটাই করবে, যেটা দিয়ে আমাদের প্রথম ML model বানানো হয়েছে।
প্রতিটা ধাপে আগে বাংলায় বোঝানো আছে **কেন** করছি, তারপর code।

**কীভাবে চালাবে (Google Colab):**
1. colab.research.google.com → File → Upload notebook → এই ফাইল
2. বাম পাশে 📁 (Files) → Upload → `BASS-II_flame_spread_microgravity_v1.0.csv`
3. উপর থেকে একটা একটা cell চালাও (Shift + Enter)

⏱ সময়: প্রায় ১.৫–২ ঘণ্টা। তাড়াহুড়ো না করে প্রতিটা output পড়ো।

## ধাপ ০ — ব্যাপারটা আসলে কী?
ISS-এ NASA একটা প্লাস্টিকের রড (PMMA) জ্বালিয়ে দেখেছে আগুন কত দ্রুত ছড়ায়।
তারা বদলেছে ৩টা জিনিস: **অক্সিজেন কত %**, **বাতাস কত জোরে আসছে**, **রড কত মোটা**।
আর মেপেছে ১টা জিনিস: **আগুন প্রতি সেকেন্ডে কত mm এগোল**।

ML-এর ভাষায়:
- যেগুলো আমরা দিই → **features (X)** = O₂, airflow, diameter
- যেটা আমরা জানতে চাই → **target (y)** = flame spread

ML model = এমন একটা "সূত্র" যেটা data দেখে নিজে শেখে X থেকে y কীভাবে আসে।

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("BASS-II_flame_spread_microgravity_v1.0.csv")
print("কয়টা সারি, কয়টা কলাম:", df.shape)
df.head()

## ধাপ ১ — শুধু দরকারি কলামগুলো দেখো
CSV-তে অনেক কলাম আছে (source, notes ইত্যাদি)। Model-এর জন্য দরকার মাত্র কয়েকটা।
`test_id` হলো ISS-এর কোন test — এটা feature না, কিন্তু পরে খুব কাজে লাগবে (ধাপ ৬)।

In [ ]:
cols = ["test_id", "o2_pct", "opposed_flow_cm_s", "diameter_cm", "flame_spread_mm_s"]
data = df[cols]
data.head(10)

## ধাপ ২ — সংখ্যাগুলো কোন পরিসরে আছে?
`describe()` প্রতিটা কলামের min, max, গড় (mean) দেখায়।
👉 খেয়াল করো: O₂ মাত্র **16.8–20.9%**, airflow **0.4–7.6 cm/s**। Model শুধু এই পরিসরেই বিশ্বাসযোগ্য — এটাকে বলে **domain**।

In [ ]:
data.describe().round(3)

In [ ]:
print("কয়টা আলাদা ISS test:", data.test_id.nunique())
print("প্রতিটা test-এ কয়টা মাপ:")
print(data.test_id.value_counts().sort_index().to_dict())

## ধাপ ৩ — চোখে দেখো (plot)
Data science-এর প্রথম নিয়ম: **model বানানোর আগে graph আঁকো।**
প্রশ্ন: O₂ বাড়লে আগুন কি দ্রুত ছড়ায়? মোটা রডে কি ধীরে?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for d, c in [(0.64, "red"), (0.95, "orange"), (1.27, "blue")]:
    s = data[data.diameter_cm == d]
    ax[0].scatter(s.o2_pct, s.flame_spread_mm_s, color=c, label=f"d = {d} cm")
    ax[1].scatter(s.opposed_flow_cm_s, s.flame_spread_mm_s, color=c, label=f"d = {d} cm")
ax[0].set(xlabel="O2 (%)", ylabel="Flame spread (mm/s)", title="Oxygen vs spread")
ax[1].set(xlabel="Airflow (cm/s)", ylabel="Flame spread (mm/s)", title="Airflow vs spread")
ax[0].legend(); ax[1].legend(); plt.show()

✍️ **নিজে লেখো (মনে মনে বা খাতায়):** লাল বিন্দুগুলো (সরু রড) কি উপরের দিকে? O₂ বাড়লে বিন্দু কি উপরে ওঠে?

## ধাপ ৪ — Correlation: কোন feature-এর সাথে target বেশি জড়িত?
Correlation −1 থেকে +1। +1 = একসাথে বাড়ে, −1 = একটা বাড়লে অন্যটা কমে, 0 = সম্পর্ক নেই।

In [ ]:
data[["o2_pct", "opposed_flow_cm_s", "diameter_cm", "flame_spread_mm_s"]].corr().round(2)["flame_spread_mm_s"]

## ধাপ ৫ — সবচেয়ে বোকা model (baseline)
যেকোনো ML-এর আগে একটা "বোকা" উত্তর ঠিক করি: **সবসময় গড় মান বলে দাও।**
আমাদের model যদি এর চেয়ে ভালো না করে, তাহলে model বৃথা।

ভুল মাপব **MAE (Mean Absolute Error)** দিয়ে = গড়ে কত mm/s ভুল।

In [ ]:
y = data.flame_spread_mm_s.values
mean_guess = y.mean()
mae_baseline = np.mean(np.abs(y - mean_guess))
print(f"সবসময় {mean_guess:.4f} mm/s বললে গড় ভুল (MAE) = {mae_baseline:.4f} mm/s")

## ধাপ ৬ — সবচেয়ে গুরুত্বপূর্ণ ধারণা: কেন test দিয়ে ভাগ করি (Grouped CV)
Model-কে পরীক্ষা করতে হয় **এমন data দিয়ে যেটা সে শেখার সময় দেখেনি** — নাহলে সেটা মুখস্থ বিদ্যা।

কিন্তু একটা ফাঁদ আছে: একই ISS test-এর মাপগুলো একে অপরের খুব কাছাকাছি।
যদি P16 test-এর ৩টা মাপ দিয়ে শেখাই আর ৪র্থটা দিয়ে পরীক্ষা করি, model সহজেই "আন্দাজ" করে ফেলবে → ফলাফল মিথ্যা ভালো দেখাবে।

✅ সমাধান: **Leave-One-Test-Out** — প্রতিবার একটা পুরো test বাদ রাখো, বাকি ১৮টা দিয়ে শেখাও, বাদ দেওয়াটা predict করো। ১৯ বার ঘুরাও।
এটাই আমাদের model-কে NASA judge-দের সামনে বিশ্বাসযোগ্য করে।

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

X = pd.DataFrame({
    "o2_pct": data.o2_pct,
    "opposed_flow_cm_s": data.opposed_flow_cm_s,
    "inv_diameter": 1 / data.diameter_cm,      # সরু রড = বড় সংখ্যা
})
groups = data.test_id
logo = LeaveOneGroupOut()

pred_lin = cross_val_predict(LinearRegression(), X, y, groups=groups, cv=logo)
print(f"Linear regression  MAE = {mean_absolute_error(y, pred_lin):.4f}   R2 = {r2_score(y, pred_lin):.2f}")
print(f"Baseline (গড়)      MAE = {mae_baseline:.4f}")

## ধাপ ৭ — ভুল উপায়ে পরীক্ষা করলে কী হয়? (নিজে দেখো)
এবার ইচ্ছা করে ভুল করি: সাধারণ random ভাগ (test না মেনে)। স্কোর কি বেশি ভালো দেখায়? সেটাই হলো "leakage"-এর ফাঁদ।

In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import ExtraTreesRegressor

et = ExtraTreesRegressor(n_estimators=300, min_samples_leaf=3, random_state=42)
pred_wrong = cross_val_predict(et, X, y, cv=KFold(5, shuffle=True, random_state=0))
pred_right = cross_val_predict(et, X, y, groups=groups, cv=logo)
print(f"❌ Random split       MAE = {mean_absolute_error(y, pred_wrong):.4f}  R2 = {r2_score(y, pred_wrong):.2f}")
print(f"✅ Leave-one-test-out MAE = {mean_absolute_error(y, pred_right):.4f}  R2 = {r2_score(y, pred_right):.2f}")

## ধাপ ৮ — ফলাফল বোঝা
- **MAE** = গড় ভুল (mm/s)। ছোট = ভালো।
- **R²** = data-র ওঠানামার কত অংশ model ব্যাখ্যা করে। 1 = নিখুঁত, 0 = গড়ের সমান, নেগেটিভ = গড়ের চেয়েও খারাপ।

নিচের graph: x = আসল মান, y = model-এর অনুমান। বিন্দুগুলো কালো দাগের যত কাছে, তত ভালো।

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(y, pred_right, edgecolor="k")
lim = [0, 0.11]; plt.plot(lim, lim, "k--")
plt.xlabel("Measured (mm/s)"); plt.ylabel("Predicted, test held out (mm/s)")
plt.title("Extra Trees — leave-one-test-out"); plt.show()

## ধাপ ৯ — নতুন অবস্থার জন্য predict + domain check
এখন সব data দিয়ে চূড়ান্ত model শেখাই, তারপর নতুন একটা অবস্থা জিজ্ঞেস করি।
কিন্তু আগে দেখি অবস্থাটা NASA-র পরীক্ষিত পরিসরের ভেতরে কি না — না হলে সতর্ক করতে হবে।

In [ ]:
et.fit(X, y)

def ask(o2, flow, diameter):
    s = data[data.diameter_cm == diameter]
    ok = (not s.empty) and s.o2_pct.min() <= o2 <= s.o2_pct.max() and s.opposed_flow_cm_s.min() <= flow <= s.opposed_flow_cm_s.max()
    x = pd.DataFrame({"o2_pct": [o2], "opposed_flow_cm_s": [flow], "inv_diameter": [1 / diameter]})
    p = et.predict(x)[0]
    print(f"O2={o2}%, flow={flow} cm/s, d={diameter} cm → {p:.4f} mm/s   {'✅ in-domain' if ok else '⚠️ OUT of domain (বিশ্বাস কম)'}")

ask(18.2, 1.67, 0.64)
ask(20.9, 3.0, 0.64)   # 0.64 cm রড কখনো 18.9%-এর বেশি O2-তে পরীক্ষা হয়নি
ask(30, 25, 0.95)      # একদম বাইরে

## ধাপ ১০ — নিজে চেষ্টা করো ✍️
1. `ask()` দিয়ে ৩টা নিজের অবস্থা দাও। কোনগুলো in-domain?
2. ধাপ ৬-এ `inv_diameter`-এর জায়গায় সরাসরি `diameter_cm` দিয়ে দেখো — MAE বাড়ে না কমে?
3. ধাপ ৭-এ `min_samples_leaf=3` বদলে `1` আর `10` দিয়ে দেখো। কোনটা ভালো? কেন মনে হয়?
4. এক লাইনে লেখো: "কেন আমরা random split ব্যবহার করি না?"

এই ৪টা পারলে তুমি team-কে model-টা নিজের ভাষায় বোঝাতে পারবে। 🎯